# LeetCode #198 — House Robber: quy hoạch động "lấy hay bỏ"

**Mức độ:** Medium · **Chủ đề:** Array, Dynamic Programming

Skeleton trên LeetCode:

```python
class Solution:
    def rob(self, nums: List[int]) -> int:
```

Mục tiêu của notebook:

1. Thấy vì sao **tham lam** (lấy nhà nhiều tiền nhất, hay lấy chẵn/lẻ xen kẽ) đều **sai**.
2. Đi từ đệ quy "lấy hay bỏ nhà cuối" → **ghi nhớ** → **bảng** → **hai biến**.
3. Định nghĩa trạng thái rõ ràng: `dp[i]` = số tiền lớn nhất khi chỉ xét `i` nhà đầu.
4. Truy vết để biết **những nhà nào** được chọn.

# 1. Bài toán là gì?

Một dãy nhà, nhà `i` có `nums[i]` tiền. Không được lấy **hai nhà liền kề** (hệ thống báo động nối giữa các nhà kề nhau). Hỏi lấy được tối đa bao nhiêu tiền?

**Constraints (theo đề):**

- `1 <= nums.length <= 100`
- `0 <= nums[i] <= 400`

---

## Ví dụ

```text
[1, 2, 3, 1]      ->  4     (nhà 0 và 2: 1 + 3)
[2, 7, 9, 3, 1]   ->  12    (nhà 0, 2, 4: 2 + 9 + 1)
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quy hoạch động (Dynamic Programming — DP)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [1]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [2]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.2 Áp dụng vào bài này

Đây chính là ví dụ "chọn hay không chọn" của **quy hoạch động** ở trên: trạng thái `best(i)` = tiền lớn nhất trên các nhà `0..i`, chuyển trạng thái `best(i) = max(best(i−1), best(i−2) + nums[i])`. Vì chỉ dùng hai trạng thái gần nhất nên bảng DP thu gọn còn **hai biến**.

# 3. Nền tảng

In [3]:
from typing import List
from functools import lru_cache
from itertools import combinations
import random, time

def is_valid(chosen):
    return all(b - a >= 2 for a, b in zip(chosen, chosen[1:]))

print(is_valid([0, 2, 4]), is_valid([0, 1, 3]))

True False


## Thử các cách tham lam

In [4]:
def greedy_max_first(nums):
    """Lấy nhà nhiều tiền nhất còn lấy được, lặp lại."""
    taken, total = set(), 0
    for i in sorted(range(len(nums)), key=lambda i: -nums[i]):
        if i - 1 not in taken and i + 1 not in taken:
            taken.add(i); total += nums[i]
    return total

def greedy_even_odd(nums):
    return max(sum(nums[0::2]), sum(nums[1::2]))

for t in [[2, 7, 9, 3, 1], [2, 3, 2], [5, 1, 1, 5]]:
    print(t, "| lấy max trước:", greedy_max_first(t), "| chẵn/lẻ:", greedy_even_odd(t))

[2, 7, 9, 3, 1] | lấy max trước: 12 | chẵn/lẻ: 12
[2, 3, 2] | lấy max trước: 3 | chẵn/lẻ: 4
[5, 1, 1, 5] | lấy max trước: 10 | chẵn/lẻ: 6


Với `[2, 3, 2]`, lấy nhà nhiều tiền nhất (3) trước cho 3, trong khi `2 + 2 = 4`. Với `[5, 1, 1, 5]`, chẵn/lẻ cho 6 nhưng `5 + 5 = 10` (nhà 0 và 3 — cách nhau **hai** nhà). Cần xét **mọi** lựa chọn một cách có hệ thống → quy hoạch động.

# 4. Baseline: thử mọi tập nhà hợp lệ

In [5]:
def rob_brute(nums: List[int]) -> int:
    n, best = len(nums), 0
    for r in range(n + 1):
        for chosen in combinations(range(n), r):
            if is_valid(chosen):
                best = max(best, sum(nums[i] for i in chosen))
    return best

print(rob_brute([1, 2, 3, 1]), rob_brute([2, 7, 9, 3, 1]))

4 12


`2^n` tập con — với `n = 100` là bất khả thi.

# 5. Ý tưởng chính: lấy hay bỏ nhà cuối cùng

Gọi `f(i)` = số tiền lớn nhất khi chỉ xét các nhà `0..i−1` (**`i` nhà đầu**). Xét nhà cuối cùng trong số đó (nhà `i − 1`):

- **Bỏ** nhà `i − 1` → phần còn lại là bài toán `f(i − 1)`.
- **Lấy** nhà `i − 1` → không được lấy nhà `i − 2` → `f(i − 2) + nums[i − 1]`.

```text
f(i) = max( f(i − 1), f(i − 2) + nums[i − 1] )
f(0) = 0                (không có nhà)
f(1) = nums[0]
đáp án = f(n)
```

## Bước 1 — Đệ quy thuần (chậm vì tính lặp)

In [6]:
calls = 0
def rob_rec(nums, i):
    global calls
    calls += 1
    if i == 0:
        return 0
    if i == 1:
        return nums[0]
    return max(rob_rec(nums, i - 1), rob_rec(nums, i - 2) + nums[i - 1])

calls = 0
print("kết quả:", rob_rec([2, 7, 9, 3, 1] * 5, 25), "| số lời gọi:", calls)

kết quả: 56 | số lời gọi: 242785


## Bước 2 — Ghi nhớ (top-down)

In [7]:
def rob_memo(nums: List[int]) -> int:
    @lru_cache(maxsize=None)
    def f(i):
        if i == 0:
            return 0
        if i == 1:
            return nums[0]
        return max(f(i - 1), f(i - 2) + nums[i - 1])
    return f(len(nums))

print(rob_memo([1, 2, 3, 1]), rob_memo([2, 7, 9, 3, 1]))

4 12


## Bước 3 — Bảng (bottom-up)

In [8]:
def rob_table(nums: List[int]) -> int:
    n = len(nums)
    dp = [0] * (n + 1)
    dp[1] = nums[0]
    for i in range(2, n + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + nums[i - 1])
    return dp[n]

print(rob_table([1, 2, 3, 1]), rob_table([2, 7, 9, 3, 1]))

4 12


## Bước 4 — Chỉ giữ hai giá trị gần nhất

`dp[i]` chỉ cần `dp[i−1]` và `dp[i−2]` → hai biến `prev2`, `prev1`.

In [9]:
def rob_two_vars(nums: List[int]) -> int:
    prev2, prev1 = 0, 0                 # dp[i-2], dp[i-1]
    for x in nums:
        prev2, prev1 = prev1, max(prev1, prev2 + x)
    return prev1

print(rob_two_vars([1, 2, 3, 1]), rob_two_vars([2, 7, 9, 3, 1]))

4 12


(Bắt đầu với `prev2 = prev1 = 0` thì không cần xử lý riêng `dp[1]`: bước đầu cho `max(0, 0 + nums[0]) = nums[0]`.)

# 6. Mô phỏng: bảng `dp` và truy vết các nhà được chọn

In [10]:
def trace_rob(nums):
    n = len(nums)
    dp = [0] * (n + 1)
    dp[1] = nums[0]
    print(f"{'i':>2} {'nhà':>4} {'tiền':>5} | {'bỏ: dp[i-1]':>11} {'lấy: dp[i-2]+tiền':>18} | dp[i]")
    print(f"{1:>2} {0:>4} {nums[0]:>5} | {'':>11} {'':>18} | {dp[1]}")
    for i in range(2, n + 1):
        skip, take = dp[i - 1], dp[i - 2] + nums[i - 1]
        dp[i] = max(skip, take)
        print(f"{i:>2} {i - 1:>4} {nums[i - 1]:>5} | {skip:>11} {take:>18} | {dp[i]}")
    chosen, i = [], n
    while i >= 1:                                  # đi ngược: dp[i] đến từ "lấy" hay "bỏ"?
        if i == 1 or dp[i] != dp[i - 1]:
            chosen.append(i - 1)
            i -= 2
        else:
            i -= 1
    print("nhà được chọn:", chosen[::-1], "→ tổng", sum(nums[j] for j in chosen))

trace_rob([2, 7, 9, 3, 1])

 i  nhà  tiền | bỏ: dp[i-1]  lấy: dp[i-2]+tiền | dp[i]
 1    0     2 |                                | 2
 2    1     7 |           2                  7 | 7
 3    2     9 |           7                 11 | 11
 4    3     3 |          11                 10 | 11
 5    4     1 |          11                 12 | 12
nhà được chọn: [0, 2, 4] → tổng 12


# 7. Phiên bản đúng format LeetCode

In [11]:
class Solution:
    def rob(self, nums: List[int]) -> int:
        prev2, prev1 = 0, 0
        for x in nums:
            prev2, prev1 = prev1, max(prev1, prev2 + x)
        return prev1

In [12]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<12} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.rob([1, 2, 3, 1]), 4)
check("Ví dụ 2", sol.rob([2, 7, 9, 3, 1]), 12)
check("Một nhà", sol.rob([5]), 5)
check("Hai nhà", sol.rob([2, 1]), 2)
check("Cách hai", sol.rob([5, 1, 1, 5]), 10)
check("Toàn 0", sol.rob([0, 0, 0]), 0)

random.seed(198)
for _ in range(2000):
    nums = [random.randint(0, 20) for _ in range(random.randint(1, 12))]
    exp = rob_brute(nums)
    assert sol.rob(nums) == exp == rob_memo(nums) == rob_table(nums), nums
print("2000 mảng ngẫu nhiên: bốn cách khớp brute force")

calls = 0
rob_rec([random.randint(0, 400) for _ in range(25)], 25)
print("đệ quy thuần với n = 25:", calls, "lời gọi | DP: 25 bước")

Ví dụ 1      PASS  4
Ví dụ 2      PASS  12
Một nhà      PASS  5
Hai nhà      PASS  2
Cách hai     PASS  10
Toàn 0       PASS  0


2000 mảng ngẫu nhiên: bốn cách khớp brute force
đệ quy thuần với n = 25: 242785 lời gọi | DP: 25 bước


# 8. Vì sao đúng? Độ phức tạp?

**Cấu trúc con tối ưu:** một cách chọn tối ưu cho `i` nhà đầu hoặc không chứa nhà `i − 1` (khi đó phần còn lại là cách chọn hợp lệ cho `i − 1` nhà đầu, và phải tối ưu — nếu không thay bằng cách tốt hơn sẽ được tổng lớn hơn), hoặc chứa nhà `i − 1` (khi đó không chứa nhà `i − 2`, phần còn lại là cách chọn cho `i − 2` nhà đầu, cũng phải tối ưu). Lấy `max` hai khả năng.

**Bài con chồng lấp:** `f(i − 1)` và `f(i − 2)` đều gọi `f(i − 3)`… — đệ quy thuần tăng như Fibonacci (`O(φ^n)`), ghi nhớ còn `O(n)`.

```text
Time:  O(n)
Space: O(1) (bản hai biến); O(n) nếu cần truy vết
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Tham lam

Xem mục 3: lấy nhà nhiều tiền nhất trước, hoặc chọn chẵn/lẻ, đều có phản ví dụ.

## Lỗi 2: Cập nhật hai biến sai thứ tự

In [13]:
def wrong_order(nums):
    prev2, prev1 = 0, 0
    for x in nums:
        prev2 = prev1                              # SAI: ghi đè prev2 trước khi dùng giá trị cũ
        prev1 = max(prev1, prev2 + x)
    return prev1

for t in [[2, 7, 9, 3, 1], [1, 2, 3, 1]]:
    print(t, "→ sai:", wrong_order(t), "| đúng:", sol.rob(t))

[2, 7, 9, 3, 1] → sai: 22 | đúng: 12
[1, 2, 3, 1] → sai: 7 | đúng: 4


Sau dòng đầu, `prev2` đã là `dp[i−1]` nên công thức thành `dp[i−1] + x` — được phép lấy hai nhà liền kề. Dùng gán bộ `prev2, prev1 = prev1, max(...)`.

## Lỗi 3: `dp[i] = dp[i − 2] + nums[i]` (luôn lấy nhà hiện tại)

In [14]:
def wrong_always_take(nums):
    n = len(nums)
    dp = [0] * n
    for i in range(n):
        dp[i] = (dp[i - 2] if i >= 2 else 0) + nums[i]
    return max(dp)

for t in [[2, 1, 1, 2], [2, 7, 9, 3, 1]]:
    print(t, "→ sai:", wrong_always_take(t), "| đúng:", sol.rob(t))

[2, 1, 1, 2] → sai: 3 | đúng: 4
[2, 7, 9, 3, 1] → sai: 12 | đúng: 12


Định nghĩa "lấy nhà `i` bắt buộc" mà chỉ nhảy từ `i − 2` thì bỏ sót khoảng cách 3 (`[2, 1, 1, 2]`: nhà 0 và 3). Hoặc thêm `dp[i − 3]`, hoặc dùng định nghĩa "xét `i` nhà đầu" với lựa chọn **bỏ**.

## Lỗi 4: Lệch chỉ số giữa `dp` (dài `n + 1`) và `nums` (dài `n`)

`dp[i]` ứng với **`i` nhà đầu**, nhà cuối của nó là `nums[i − 1]`. Viết `nums[i]` sẽ lệch một nhà (và vượt mảng ở `i = n`).

# 10. Mẹo nhận diện pattern

- **Chọn các phần tử, có ràng buộc "không chọn hai cái kề nhau"** → DP lấy/bỏ phần tử cuối.
- **Công thức chỉ nhìn 1–2 bước trước** → tối ưu bộ nhớ còn vài biến.
- **Cần biết đã chọn những gì** → giữ bảng đầy đủ rồi truy vết ngược.

Bài liên quan: **#213 House Robber II** (nhà xếp vòng tròn), **#337 House Robber III** (trên cây), **#740 Delete and Earn**, **#70 Climbing Stairs** (Easy), **#2560 House Robber IV** (Medium, tìm nhị phân + tham lam).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Lập bảng `dp` cho `[6, 1, 2, 9, 4, 3]` và truy vết các nhà được chọn.

## Bài 2 — Điền chỗ trống (hai biến)

In [15]:
def exercise_2_rob(nums: List[int]) -> int:
    prev2, prev1 = 0, 0
    for x in nums:
        # TODO: một dòng gán bộ
        pass
    return prev1


# assert exercise_2_rob([1, 2, 3, 1]) == 4
# assert exercise_2_rob([2, 7, 9, 3, 1]) == 12
# assert exercise_2_rob([5, 1, 1, 5]) == 10

## Bài 3 — Biến thể: trả về **danh sách chỉ số** nhà được chọn (tăng dần)

Nếu có nhiều đáp án, trả bất kỳ (assert kiểm tra tính hợp lệ và tổng).

In [16]:
def exercise_3_rob_houses(nums: List[int]) -> List[int]:
    # TODO
    pass


# r3 = exercise_3_rob_houses([2, 7, 9, 3, 1]); assert is_valid(r3) and sum([2, 7, 9, 3, 1][i] for i in r3) == 12
# r3b = exercise_3_rob_houses([5, 1, 1, 5]); assert r3b == [0, 3]

## Bài 4 — Biến thể: Delete and Earn (LeetCode #740)

Chọn số `x` được `x` điểm, nhưng mọi phần tử bằng `x − 1` và `x + 1` bị xóa. Gợi ý: gom `points[v] = v · (số lần v xuất hiện)`; các **giá trị** `v` và `v + 1` không được chọn cùng lúc → chính là House Robber trên mảng `points[0..max]`.

In [17]:
def exercise_4_delete_and_earn(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_delete_and_earn([3, 4, 2]) == 6
# assert exercise_4_delete_and_earn([2, 2, 3, 3, 3, 4]) == 9
# assert exercise_4_delete_and_earn([1]) == 1

## Bài 5 — Mini challenge: House Robber III (LeetCode #337) — nhà xếp thành cây nhị phân

Không lấy hai nhà có quan hệ **cha – con**. Gợi ý: DFS trả về cặp `(lấy_nút_này, không_lấy_nút_này)`.

In [18]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right


def exercise_5_rob_tree(root) -> int:
    # TODO
    pass


# assert exercise_5_rob_tree(TreeNode(3, TreeNode(2, None, TreeNode(3)), TreeNode(3, None, TreeNode(1)))) == 7
# assert exercise_5_rob_tree(TreeNode(3, TreeNode(4, TreeNode(1), TreeNode(3)), TreeNode(5, None, TreeNode(1)))) == 9
# assert exercise_5_rob_tree(None) == 0

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_rob([6, 1, 2, 9, 4, 3])` sau khi tự làm.

**Bài 3**
- Mức 1: dựng bảng `dp` đầy đủ; đi ngược từ `i = n`: nếu `dp[i] != dp[i−1]` thì nhà `i−1` được lấy, nhảy `i −= 2`; ngược lại `i −= 1`.

**Bài 4**
- Mức 1: `points = [0] * (max(nums) + 1)`; `points[v] += v`; trả `rob(points)`.

**Bài 5**
- Mức 1: `dfs(None) = (0, 0)`.
- Mức 2: `lấy = node.val + khônglấy(trái) + khônglấy(phải)`; `không_lấy = max(trái) + max(phải)`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [19]:
trace_rob([6, 1, 2, 9, 4, 3])

 i  nhà  tiền | bỏ: dp[i-1]  lấy: dp[i-2]+tiền | dp[i]
 1    0     6 |                                | 6
 2    1     1 |           6                  1 | 6
 3    2     2 |           6                  8 | 8
 4    3     9 |           8                 15 | 15
 5    4     4 |          15                 12 | 15
 6    5     3 |          15                 18 | 18
nhà được chọn: [0, 3, 5] → tổng 18


## Đáp án Bài 2

In [20]:
def answer_2_rob(nums: List[int]) -> int:
    return rob_two_vars(nums)

assert answer_2_rob([1, 2, 3, 1]) == 4
assert answer_2_rob([2, 7, 9, 3, 1]) == 12
assert answer_2_rob([5, 1, 1, 5]) == 10
print("Bài 2 PASS (giống rob_two_vars ở mục 5)")

Bài 2 PASS (giống rob_two_vars ở mục 5)


## Đáp án Bài 3

In [21]:
def answer_3_rob_houses(nums: List[int]) -> List[int]:
    n = len(nums)
    dp = [0] * (n + 1)
    dp[1] = nums[0]
    for i in range(2, n + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + nums[i - 1])
    chosen, i = [], n
    while i >= 1:
        if i == 1 or dp[i] != dp[i - 1]:
            chosen.append(i - 1)
            i -= 2
        else:
            i -= 1
    return chosen[::-1]

r3 = answer_3_rob_houses([2, 7, 9, 3, 1]); assert is_valid(r3) and sum([2, 7, 9, 3, 1][i] for i in r3) == 12
r3b = answer_3_rob_houses([5, 1, 1, 5]); assert r3b == [0, 3]
for _ in range(2000):
    nums = [random.randint(0, 20) for _ in range(random.randint(1, 12))]
    r = answer_3_rob_houses(nums)
    assert is_valid(r) and sum(nums[i] for i in r) == sol.rob(nums)
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [22]:
def answer_4_delete_and_earn(nums: List[int]) -> int:
    points = [0] * (max(nums) + 1)
    for v in nums:
        points[v] += v
    return sol.rob(points)

def brute_740(nums):
    from collections import Counter
    c = Counter(nums); vals = sorted(c)
    best = 0
    for r in range(len(vals) + 1):
        for pick in combinations(vals, r):
            if all(b - a >= 2 for a, b in zip(pick, pick[1:])):
                best = max(best, sum(v * c[v] for v in pick))
    return best

assert answer_4_delete_and_earn([3, 4, 2]) == 6
assert answer_4_delete_and_earn([2, 2, 3, 3, 3, 4]) == 9
assert answer_4_delete_and_earn([1]) == 1
for _ in range(1000):
    nums = [random.randint(1, 8) for _ in range(random.randint(1, 8))]
    assert answer_4_delete_and_earn(nums) == brute_740(nums)
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [23]:
def answer_5_rob_tree(root) -> int:
    def dfs(node):
        if node is None:
            return 0, 0
        lt, ln = dfs(node.left)
        rt, rn = dfs(node.right)
        take = node.val + ln + rn
        skip = max(lt, ln) + max(rt, rn)
        return take, skip
    return max(dfs(root))

assert answer_5_rob_tree(TreeNode(3, TreeNode(2, None, TreeNode(3)), TreeNode(3, None, TreeNode(1)))) == 7
assert answer_5_rob_tree(TreeNode(3, TreeNode(4, TreeNode(1), TreeNode(3)), TreeNode(5, None, TreeNode(1)))) == 9
assert answer_5_rob_tree(None) == 0

def path_tree(vals):                                # cây lệch = một dãy nhà → phải trùng House Robber gốc
    root = None
    for v in reversed(vals):
        root = TreeNode(v, None, root)
    return root

for _ in range(1000):
    vals = [random.randint(0, 20) for _ in range(random.randint(1, 12))]
    assert answer_5_rob_tree(path_tree(vals)) == sol.rob(vals)
print("Bài 5 PASS (cây lệch trùng kết quả bài gốc)")

Bài 5 PASS (cây lệch trùng kết quả bài gốc)

# 14. Tổng kết

> `dp[i] = max(dp[i−1], dp[i−2] + nums[i−1])` — bỏ hoặc lấy nhà cuối. Chỉ cần hai biến: `prev2, prev1 = prev1, max(prev1, prev2 + x)`. `O(n)` thời gian, `O(1)` bộ nhớ. Tham lam đều sai.

---

## Checklist trước khi nộp

- [ ] Có lựa chọn **bỏ** nhà hiện tại?
- [ ] Gán bộ khi cập nhật hai biến?
- [ ] Đúng với một nhà, hai nhà?
- [ ] Chỉ số `dp[i]` ↔ `nums[i − 1]` nhất quán?